# Pretrained Image Classification and Transfer Learning with PyTorch

Explore image classification with **VGG16**, **ResNet50**, **AlexNet**, **ResNet101**, and **MobileNetV2**, then adapt an ImageNet-pretrained **ResNet18** to **CIFAR-10** using feature extraction.

This notebook covers:

1. **ImageNet inference:** compare the top five predictions from five pretrained architectures.
2. **Transfer learning:** freeze a ResNet18 feature extractor and train a new ten-class prediction head.
3. **Evaluation:** inspect classification scores, training/test accuracy, and save/reload the adapted model.

The examples run with `torch` and `torchvision`. On first execution, model weights and CIFAR-10 are downloaded. GPU acceleration is optional. Execute the notebook from top to bottom to reproduce the workflow.


## Environment setup

Install the dependencies before running. To use NVIDIA CUDA, choose a matching PyTorch build using the instructions at [pytorch.org](https://pytorch.org/get-started/locally/).

In [ ]:
# Uncomment this command if the required packages are missing:
# %pip install torch torchvision pillow


## Imports and hardware selection

`torch` implements tensor computation and optimization, `torchvision` supplies pretrained networks and CIFAR-10, and Pillow (`PIL`) loads images.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from PIL import Image

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
SAMPLE_URL = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"

def select_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = select_device()
print("PyTorch:", torch.__version__, "| Computation device:", device)

## Input image

Use a local JPG/PNG file by setting `IMAGE_PATH`, or leave it as `None` to fetch the example dog image hosted in the PyTorch Hub repository.

In [ ]:
def get_image(image_path):
    """Use the supplied image or download a sample photo once."""
    if image_path is not None:
        path = Path(image_path)
        if not path.is_file():
            raise FileNotFoundError(f"Image not found: {path}")
        return path
    path = Path("data") / "dog.jpg"
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        print(f"Downloading example image to {path} ...")
        try:
            urlretrieve(SAMPLE_URL, path)
        except Exception as exc:
            raise RuntimeError(
                "Could not download the sample image. Run with --image path/to/photo.jpg"
            ) from exc
    return path

IMAGE_PATH = None  # Example: r"C:\Users\YourName\Pictures\dog.jpg"
image_path = get_image(IMAGE_PATH)
print("Image:", image_path)
display(Image.open(image_path).convert("RGB"))

## ImageNet model comparison

Five architectures are loaded with their respective pretrained weights and preprocessing transforms. Each produces 1,000 ImageNet class scores (logits); applying softmax gives model probabilities and `topk(5)` selects the five highest-scoring labels. Models load sequentially to reduce peak memory use.

**Note:** This section performs inference using existing weights; it does not update model parameters.

In [ ]:
def predict_top5(image_path, device):
    """Run five pretrained ImageNet classifiers, loading one at a time."""
    specifications = [
        ("VGG16", models.vgg16, models.VGG16_Weights.DEFAULT),
        ("ResNet50", models.resnet50, models.ResNet50_Weights.DEFAULT),
        ("AlexNet", models.alexnet, models.AlexNet_Weights.DEFAULT),
        ("ResNet101", models.resnet101, models.ResNet101_Weights.DEFAULT),
        ("MobileNetV2", models.mobilenet_v2, models.MobileNet_V2_Weights.DEFAULT),
    ]
    picture = Image.open(image_path).convert("RGB")
    results = {}
    for name, constructor, weights in specifications:
        print(f"\nLoading {name} (first use may download weights) ...")
        network = constructor(weights=weights).to(device)
        network.eval()  # Disable training-time behavior
        image_tensor = weights.transforms()(picture).unsqueeze(0).to(device)

        with torch.inference_mode():
            probabilities = torch.softmax(network(image_tensor), dim=1)[0]
            scores, indices = torch.topk(probabilities, k=5)

        classes = weights.meta["categories"]
        top5 = [(classes[i], float(score)) for score, i in
                zip(scores.cpu().tolist(), indices.cpu().tolist())]
        results[name] = top5
        print(f"{name} — top 5 ImageNet predictions")
        for rank, (label, score) in enumerate(top5, 1):
            print(f"  {rank}. {label:30s} {score * 100:6.2f}%")

        # Discard each model before loading the next large set of weights.
        del network, image_tensor, probabilities
        if device.type == "cuda":
            torch.cuda.empty_cache()
    return results

results = predict_top5(image_path, device)

### Understanding classification scores

- The top-ranked category is the model's prediction, not a verified ground-truth label.
- Softmax scores are model probabilities, not guaranteed calibrated confidence estimates.
- Architectures and pretrained weight variants can return different labels.
- Examine both the predicted categories and score distributions when comparing networks.

The next cell presents a compact top-1 summary.

In [ ]:
print(f"{'Model':<16} {'Top prediction':<30} {'Score':>8}")
print('-' * 58)
for model_name, top5 in results.items():
    label, score = top5[0]
    print(f"{model_name:<16} {label:<30} {score * 100:>7.2f}%")

## CIFAR-10 transfer learning with ResNet18

CIFAR-10 contains 50,000 training images and 10,000 test images distributed among ten classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, and truck. The default configuration trains and evaluates on smaller subsets for quicker experimentation.

**Feature-extraction approach**

1. Initialize ResNet18 with ImageNet-pretrained weights.
2. Freeze the pretrained feature extractor (`requires_grad=False`).
3. Replace the final 1,000-class layer with a ten-class linear classifier.
4. Optimize the replacement layer using CIFAR-10 images and labels.
5. Evaluate on held-out images, inspect predictions, and save model parameters.

The original 32×32 inputs are resized to 160×160. This matches the chosen network's input pipeline at a lower computational cost, but does not restore details absent in the original images.

### Training configuration

The training function downloads CIFAR-10 if needed, creates seeded subsets and data loaders, initializes the pretrained network, and logs loss plus train/test accuracy each epoch.

Default settings: **3 epochs**, **5,000 training images**, **1,000 test images**, **batch size 64**. Increase the data size or number of epochs for a more thorough experiment (with higher compute cost).

In [ ]:
def train_cifar10_transfer(device, data_dir="data", epochs=3, train_samples=5000,
                           test_samples=1000, batch_size=64, output="cifar10_resnet18.pth"):
    """Freeze pretrained ResNet18 features and train a new CIFAR-10 classifier."""
    torch.manual_seed(42)
    train_transform = transforms.Compose([
        transforms.Resize((160, 160)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    test_transform = transforms.Compose([
        transforms.Resize((160, 160)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    train_set = datasets.CIFAR10(root=data_dir, train=True, download=True,
                                 transform=train_transform)
    test_set = datasets.CIFAR10(root=data_dir, train=False, download=True,
                                transform=test_transform)
    class_names = train_set.classes

    # Use deterministic subsets for a reasonably quick run.
    train_indices = torch.randperm(len(train_set), generator=torch.Generator().manual_seed(42))[
        :min(train_samples, len(train_set))].tolist()
    test_indices = torch.randperm(len(test_set), generator=torch.Generator().manual_seed(43))[
        :min(test_samples, len(test_set))].tolist()
    train_loader = DataLoader(Subset(train_set, train_indices), batch_size=batch_size,
                              shuffle=True, num_workers=0)
    test_loader = DataLoader(Subset(test_set, test_indices), batch_size=batch_size,
                             shuffle=False, num_workers=0)

    net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for parameter in net.parameters():
        parameter.requires_grad = False  # Keep learned ImageNet visual features
    net.fc = nn.Linear(net.fc.in_features, len(class_names))  # 1000 -> 10 classes
    net = net.to(device)
    criterion = nn.CrossEntropyLoss()  # Expects raw logits, not softmax
    optimizer = torch.optim.Adam(net.fc.parameters(), lr=1e-3)

    for epoch in range(1, epochs + 1):
        net.eval()  # Keep BatchNorm statistics frozen in the backbone.
        running_loss, train_correct, seen = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = net(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            train_correct += (logits.argmax(dim=1) == labels).sum().item()
            seen += images.size(0)

        # Compute test accuracy without gradients.
        net.eval()
        correct, total = 0, 0
        with torch.inference_mode():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                predictions = net(images).argmax(dim=1)
                correct += (predictions == labels).sum().item()
                total += labels.size(0)
        print(f"Epoch {epoch:2d}/{epochs} | train loss {running_loss / seen:.4f} "
              f"| train accuracy {100 * train_correct / seen:.2f}% "
              f"| test accuracy {100 * correct / total:.2f}%")

    checkpoint = {
        "state_dict": net.state_dict(),
        "classes": class_names,
        "architecture": "resnet18",
        "image_size": 160,
    }
    torch.save(checkpoint, output)
    print(f"Saved the transfer-learned model to {output}")

    # Display predictions for one held-out test image.
    image, true_label = test_set[test_indices[0]]
    with torch.inference_mode():
        probabilities = net(image.unsqueeze(0).to(device)).softmax(dim=1)[0]
        top_scores, top_indices = probabilities.topk(5)
    print(f"\nExample CIFAR-10 image | true label: {class_names[true_label]}")
    for rank, (idx, score) in enumerate(zip(top_indices.tolist(), top_scores.tolist()), 1):
        print(f"  {rank}. {class_names[idx]:15s} {score * 100:6.2f}%")
    return net

EPOCHS = 3
TRAIN_SAMPLES = 5000
TEST_SAMPLES = 1000
BATCH_SIZE = 64

transfer_model = train_cifar10_transfer(
    device=device,
    epochs=EPOCHS,
    train_samples=TRAIN_SAMPLES,
    test_samples=TEST_SAMPLES,
    batch_size=BATCH_SIZE,
    output="cifar10_resnet18.pth",
)

## Checkpoint validation

Reload the saved state dictionary into a ResNet18 configured with the same ten-class classification head. The checkpoint also stores class names and image size.

In [ ]:
checkpoint = torch.load("cifar10_resnet18.pth", map_location="cpu", weights_only=True)
restored = models.resnet18(weights=None)
restored.fc = nn.Linear(restored.fc.in_features, len(checkpoint["classes"]))
restored.load_state_dict(checkpoint["state_dict"])
restored.eval()
print("Loaded:", checkpoint["architecture"])
print("Classes:", checkpoint["classes"])
print("Input size:", checkpoint["image_size"], "×", checkpoint["image_size"])


## Discussion and reproducibility

**ImageNet inference:** The pretrained architectures predict one of 1,000 ImageNet categories. Each model uses its weight-specific preprocessing, and softmax converts its output into a comparable probability distribution. The top-five labels offer a view of alternative category predictions.

**Feature extraction:** A pretrained ResNet18 backbone supplies fixed visual representations while a new linear head learns the ten CIFAR-10 categories. Only that head's weights are updated during optimization.

**Interpreting the outputs:** Inspect the printed predictions, epoch losses, training accuracy, and held-out test accuracy after running the cells. No metrics are included here because measured results depend on the execution environment and training run.

**Limitations:** CIFAR-10 images are low resolution, softmax scores can be miscalibrated, and the default evaluation uses a 1,000-image test subset. Training with frozen features is less flexible than fine-tuning the entire network, but is cheaper to compute.